## imports

In [ ]:
from schema.mpanze_paw_tracking_refactor import mpanze_paw_tracking_refactor as pt
from schema.mpanze_exp_refactor import mpanze_exp_refactor as exp
from schema.mpanze_widefield_refactor import mpanze_widefield_refactor as wf
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
from tqdm.autonotebook import tqdm
from pathlib import Path
import pandas as pd
from aeon.distances import sbd_distance
from aeon.distances import sbd_pairwise_distance
from time import perf_counter
import seaborn as sns
import cv2
from matplotlib import rc

from scipy.stats import false_discovery_control

import pandas as pd
from util.allen_utils import load_allen, overlay_allen

import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr

group_colors = {"Sham":'#BBBBBB', "Stroke + training": '#4477AA', "Stroke": '#AA3377'}  # tol bright 2
stroke_groups = (exp.StrokeGroup.proj(stroke_group='group')).fetch(format='frame').reset_index().filter(items=['mouse_id', 'stroke_group']).set_index('mouse_id')
stroke_groups["stroke_group"] = stroke_groups["stroke_group"].replace({'Rehab': 'Stroke + training'})
phases = (exp.ExperimentalPhase * exp.DaysFromStrokeNorm).fetch(format='frame').reset_index().filter(items=['mouse_id', "days_from_stroke_norm", 'phase']).rename(columns={"days_from_stroke_norm": "day"}).set_index(['mouse_id', "day"])

p_datasets = Path("~/neurophys_3/r_outputs/twin_grasps/datasets").expanduser()
p_datasets.mkdir(parents=True, exist_ok=True)
p_figures = Path("~/neurophys_3/r_outputs/twin_grasps/figures").expanduser()
p_figures.mkdir(parents=True, exist_ok=True)


figure definitions - manuscript

In [ ]:
# set font to Arial
rc('font',**{'family':'sans-serif','sans-serif':['Arial']})
# set font sizes to 12 for figures
rc('font', size=12)          # controls default text sizes
rc('axes', titlesize=12)     # fontsize of the axes title
rc('axes', labelsize=12)    # fontsize of the x and y labels
rc('xtick', labelsize=12)    # fontsize of the tick labels
rc('ytick', labelsize=12)    # fontsize of the tick labels
rc('legend', fontsize=10)    # legend fontsize
rc('figure', titlesize=12)  # fontsize of the figure title

# set line width to 1
rc('lines', linewidth=1)

# set dpi to 600 for figures
rc('figure', dpi=600)

# svg font type shenanigans
rc('svg', fonttype='none')

fontsize_small = 10
fontsize_medium = 12
fontsize_large = 14

# define conversion factor from inches to cm for convenience
cm = 1/2.54 * 1.5 # (scale larger for Manuscript)

# color palette for cohorts
group_colors = {'Sham':'#BBBBBB', 'Stroke':'#4477AA', 'Stroke + training':'#AA3377'}

## Rewaded grasp - windows dataset creation

### experimental info dataset

In [ ]:
keys = (
    pt.MovementSegmentation
    * exp.DaysFromStrokeNorm
    * exp.ExperimentalPhase
    * exp.StrokeGroup.proj(stroke_group='group')
    & "phase != 'Learning'"
    & "stroke_group != 'Learning'"
    & [f"days_from_stroke_norm = {d}" for d in [-3, -2, -1, 3, 7, 14, 21, 28]]
    & "mouse_id > 40"
).fetch("KEY")
print(f"Found {len(keys)} sessions")

dataset_info = (
    (
        exp.StrokeGroup.proj(stroke_group='group')
        * exp.ExperimentalPhase
        * exp.DaysFromStrokeNorm
        & keys
    )
    .fetch(format='frame')
    .reset_index()
    .filter(["mouse_id", "days_from_stroke_norm", "phase", "stroke_group"])
    .rename(columns={"days_from_stroke_norm": "day"})
    .set_index(["mouse_id", "day"])        
)
# rename groups
dataset_info["stroke_group"] = dataset_info["stroke_group"].replace({"Rehab": "Stroke + training"})
dataset_info["phase"] = dataset_info["phase"].replace({"Expert":"Pre", "Early":"Post Early", "Late":"Post Late"})
# create categorical variables - they work better with R for stats
dataset_info["p"] = pd.Categorical(dataset_info["phase"], categories=["Pre", "Post Early", "Post Late"], ordered=True)
dataset_info["g"] = pd.Categorical(dataset_info["stroke_group"], categories=["Sham", "Stroke", "Stroke + training"], ordered=True)
# day catergorical variable should be converted to string
dataset_info["d"] = pd.Categorical(dataset_info.index.get_level_values("day").astype(str), categories=[str(d) for d in [-3, -2, -1, 3, 7, 14, 21, 28]], ordered=True)
display(dataset_info)
dataset_info.to_pickle(p_datasets / "dataset_info.pkl")

###  Coordinate-based, ipsi only 300ms window

In [ ]:
# create coordinate matrix for all epochs
window_size = int(300 * 150 / 1000) # 300 ms window at 150 Hz
window_pre = int(50 * 150 / 1000)  # 50 ms pre
print(f"Using window size of {window_size} frames ({window_size/150*1000:.1f} ms)")

keys = (
    pt.MovementSegmentation
    * exp.DaysFromStrokeNorm
    * exp.ExperimentalPhase
    * exp.StrokeGroup.proj(stroke_group='group')
    & "phase != 'Learning'"
    & "stroke_group != 'Learning'"
    & [f"days_from_stroke_norm = {d}" for d in [-3, -2, -1, 3, 7, 14, 21, 28]]
    & "mouse_id > 37"
).fetch("KEY")
print(f"Found {len(keys)} sessions")

# iterate over keys
index = []
row = []
for key in tqdm(keys):
    # get ipsi key
    key_ipsi = (pt.PawRecording.Hand & dict(**key, side='ipsi')).fetch1("KEY")
    # fetch coordinate matrix
    coordinate_matrix, names = (pt.FilteredDLC.Hand & key_ipsi).fetch_coordinate_matrix()
    # fetch rewarded epochs
    epoch_ids, epoch_starts = (
        pt.MovementSegmentation.Epoch
        * pt.EpochClassification.Epoch.proj("epoch_class")
        & key_ipsi
        & "epoch_class = 'rewarded'"
        ).fetch("epoch_id", "start_time")
    # fetch synchronisation 
    frame_timestamps = (pt.Synchronisation.Hand & key_ipsi).fetch1("frame_timestamps")

    # iterate over epochs
    d = (exp.DaysFromStrokeNorm & key).fetch1("days_from_stroke_norm")
    for epoch_id, epoch_start in zip(epoch_ids, epoch_starts):
        # find start frame
        start_frame = np.searchsorted(frame_timestamps, epoch_start)
        # extract window
        if start_frame + window_size >= coordinate_matrix.shape[0]:
            continue
        if start_frame - window_pre < 0:
            continue
        window = coordinate_matrix[start_frame-window_pre:start_frame+window_size]
        for window_idx in range(window.shape[0]):
            index.append([key["mouse_id"], d, epoch_id, window_idx])
            row.append(window[window_idx])
index = pd.MultiIndex.from_tuples(index, names=["mouse_id", "day", 'epoch_id', 'time'])
dataset_coordinates_ipsi_300ms = pd.DataFrame(row, index=index, columns=names)
print(f"Extracted coordinates for {dataset_coordinates_ipsi_300ms.shape[0]} epochs with {dataset_coordinates_ipsi_300ms.shape[1]} features each")
display(dataset_coordinates_ipsi_300ms.head())
# save dataset
dataset_coordinates_ipsi_300ms.to_pickle(p_datasets / "dataset_coordinates_ipsi_300ms.pkl")

### coordinate-based, contra only, 300ms window

In [ ]:
# create coordinate matrix for all epochs
window_size = int(300 * 150 / 1000) # 300 ms window at 150 Hz
window_pre = int(50 * 150 / 1000)  # 50 ms pre
print(f"Using window size of {window_size} frames ({window_size/150*1000:.1f} ms)")

keys = (
    pt.MovementSegmentation
    * exp.DaysFromStrokeNorm
    * exp.ExperimentalPhase
    * exp.StrokeGroup.proj(stroke_group='group')
    & "phase != 'Learning'"
    & "stroke_group != 'Learning'"
    & [f"days_from_stroke_norm = {d}" for d in [-3, -2, -1, 3, 7, 14, 21, 28]]
    & "mouse_id > 37"
).fetch("KEY")
print(f"Found {len(keys)} sessions")

# iterate over keys
index = []
row = []
for key in tqdm(keys):
    # get contra and ipsi keys
    key_contra = (pt.PawRecording.Hand & dict(**key, side='contra')).fetch1("KEY")
    key_ipsi = (pt.PawRecording.Hand & dict(**key, side='ipsi')).fetch1("KEY")
    # fetch coordinate matrix
    coordinate_matrix, names = (pt.FilteredDLC.Hand & key_ipsi).fetch_coordinate_matrix()
    # fetch rewarded epochs, still align to ipsi rewarded movements
    epoch_ids, epoch_starts = (
        pt.MovementSegmentation.Epoch
        * pt.EpochClassification.Epoch.proj("epoch_class")
        & key_ipsi
        & "epoch_class = 'rewarded'"
        ).fetch("epoch_id", "start_time")
    # fetch synchronisation 
    frame_timestamps = (pt.Synchronisation.Hand & key_contra).fetch1("frame_timestamps")

    # iterate over epochs
    d = (exp.DaysFromStrokeNorm & key).fetch1("days_from_stroke_norm")
    for epoch_id, epoch_start in zip(epoch_ids, epoch_starts):
        # find start frame
        start_frame = np.searchsorted(frame_timestamps, epoch_start)
        # extract window
        if start_frame + window_size >= coordinate_matrix.shape[0]:
            continue
        if start_frame - window_pre < 0:
            continue
        window = coordinate_matrix[start_frame-window_pre:start_frame+window_size]
        for window_idx in range(window.shape[0]):
            index.append([key["mouse_id"], d, epoch_id, window_idx])
            row.append(window[window_idx])
index = pd.MultiIndex.from_tuples(index, names=["mouse_id", "day", 'epoch_id', 'time'])
dataset_coordinates_contra_300ms = pd.DataFrame(row, index=index, columns=names)
print(f"Extracted coordinates for {dataset_coordinates_contra_300ms.shape[0]} epochs with {dataset_coordinates_contra_300ms.shape[1]} features each")
display(dataset_coordinates_contra_300ms.head())
# save dataset
dataset_coordinates_contra_300ms.to_pickle(p_datasets / "dataset_coordinates_contra_300ms.pkl")

## Compute templates and distances

In [ ]:
dataset_name = "dataset_coordinates_ipsi_300ms"
dataset = pd.read_pickle(p_datasets / f"{dataset_name}.pkl")

from aeon.distances import euclidean_distance

index = []
rows = []
for mouse, df_mouse in tqdm(dataset.groupby(level='mouse_id'), desc='Computing distances to template'):
    # compute mean pre-stroke trajectory
    df_pre = df_mouse.query("day < 0")
    grasp_mean = df_pre.groupby("time").mean().to_numpy()
    for day, df_day in df_mouse.groupby(level='day'):
        for epoch_id, df_epoch in df_day.groupby(level='epoch_id'):
            grasp_to_compare = df_epoch.to_numpy()
            d = sbd_distance(grasp_mean, grasp_to_compare, standardize=False)
            # d = euclidean_distance(grasp_mean, grasp_to_compare)
            index.append([mouse, day, epoch_id])
            rows.append([d])
df_distances = pd.DataFrame(rows, index=pd.MultiIndex.from_tuples(index, names=['mouse_id', 'day', 'epoch_id']), columns=['distance'])
df_distances.to_pickle(p_datasets / f"{dataset_name}_distances.pkl")
df_distances.head()

In [ ]:
# load dataset back in
dataset_name = "dataset_coordinates_ipsi_300ms"
df_distances = pd.read_pickle(p_datasets / f"{dataset_name}_distances.pkl")
df_distances

quantile_threshold = 0.5
df_pre = df_distances.query("day < 0")
thresholds = df_pre.groupby("mouse_id")['distance'].quantile(quantile_threshold)

# threshold the data
df_distances = df_distances.join(thresholds.rename("threshold"), on='mouse_id')
df_distances["is_similar"] = df_distances["distance"] <= df_distances["threshold"]

### supplementary figure 5 A, C - compare distance distributions pre-vs-post stroke

In [ ]:
# display(stroke_groups)
f, ax = plt.subplots(2, 1, figsize=(3, 4.5), sharey=True)


df_summary = df_distances.join(stroke_groups, on='mouse_id').groupby(["stroke_group", "day"])["distance"].agg(
    ["count", "median", lambda x: np.percentile(x, 25), lambda x: np.percentile(x, 75)]
)
df_summary.to_csv(p_figures / f"supplementary_figure_3_distance_summary_{dataset_name}.csv")
df_summary_is_similar = df_distances.query("is_similar==True").join(stroke_groups, on='mouse_id').groupby(["stroke_group", "day"])["distance"].agg(
    ["count", "median", lambda x: np.percentile(x, 25), lambda x: np.percentile(x, 75)]
)
df_summary_is_similar.to_csv(p_figures / f"supplementary_figure_3_distance_summary_is_similar_{dataset_name}.csv")

display(df_summary.groupby("stroke_group")["count"].sum())
display(df_summary_is_similar.groupby("stroke_group")["count"].sum())

sns.boxplot(data=df_distances.join(stroke_groups, on='mouse_id').reset_index(),
            x='day', y='distance', hue='stroke_group',hue_order=["Sham", "Stroke", "Stroke + training"], palette=group_colors, 
            ax=ax[0], showfliers=False, fill=False, legend=False)
sns.stripplot(data=df_distances.join(stroke_groups, on='mouse_id').reset_index(),
            x='day', y='distance', hue='stroke_group',hue_order=["Sham", "Stroke", "Stroke + training"], palette=group_colors, 
            ax=ax[0], dodge=True, alpha=0.1, size=3, jitter=0.2, legend=False)

sns.boxplot(data=df_distances.query("is_similar==True").join(stroke_groups, on='mouse_id').reset_index(),
            x='day', y='distance', hue='stroke_group',hue_order=["Sham", "Stroke", "Stroke + training"], palette=group_colors, 
            ax=ax[1], showfliers=False, fill=False, legend=False)
sns.stripplot(data=df_distances.query("is_similar==True").join(stroke_groups, on='mouse_id').reset_index(),
            x='day', y='distance', hue='stroke_group',hue_order=["Sham", "Stroke", "Stroke + training"], palette=group_colors, 
            ax=ax[1], dodge=True, alpha=0.1, size=3, jitter=0.2, legend=False)

ax[0].set_xlabel("Days from stroke")
ax[1].set_xlabel("Days from stroke")
ax[0].set_ylabel("SBD distance (a.u.)")
ax[1].set_ylabel("SBD distance (a.u.)")
ax[0].set_title("All rewarded grasps")
ax[1].set_title("Most similar grasps")
# remove legend title
# ax[0].legend_.set_title(None)
f.suptitle("Distance to pre-stroke template")
f.tight_layout()
plt.show()
p_out = Path("~/neurophys_3/r_outputs/twin_grasps").expanduser()
p_out.mkdir(parents=True, exist_ok=True)
f.savefig(p_out / f"distance_comparison_{dataset_name}.png", dpi=600)
f.savefig(p_out / f"distance_comparison_{dataset_name}_contra.svg", dpi=600)

### supplementary figure 5 B, D - compute response maps

In [ ]:
# load dataset of ROI responses and continuous response maps
# start with continuous reponse maps
window = np.arange(0,20) # 20 frames after grasp onset (1 second)
wf_param_id = 6
index = []
responses = []
for session, df_session in tqdm(df_distances.query("is_similar==True").groupby(["mouse_id", "day"])):
    # get keys for this day
    key = dict(mouse_id=session[0], days_from_stroke_norm=session[1])
    key = (wf.ImageProcessing2 * exp.DaysFromStrokeNorm * pt.PawRecording.Hand & dict(**key, wf_param_id=wf_param_id, side='ipsi')).fetch1("KEY")

    # if session[0] != 42:
    #     continue
    
    # fetch widefield data
    u, svt, h, w = (wf.ImageProcessing2 & key).load_components(svt_baseline=True)
    frame_timestamps = (wf.Synchronisation & key).fetch1("frame_timestamps_blue")

    # fetch registration matrix
    M = (wf.RescaledAllenRegistration2 & key).fetch1("allen_matrix_rescaled")
    handedness = (exp.Handedness & key).fetch1("handedness")

    # get epoch starts
    restriction = df_session.reset_index().filter(["mouse_id", "day", "epoch_id"]).rename(columns={"day": "days_from_stroke_norm"})
    start_times, epoch_ids = (pt.MovementSegmentation.Epoch * exp.DaysFromStrokeNorm & restriction & key).fetch("start_time", "epoch_id", order_by="epoch_id")

    # iterate over epochs
    for start_time, epoch_id in zip(start_times, epoch_ids):
        start_frame = np.searchsorted(frame_timestamps, start_time)
        svt_window = svt[:, start_frame + window].mean(axis=1).reshape(-1,1)
        response_map = (u @ svt_window).T.reshape(h, w)

        # register and flip if necessary
        response_map = cv2.warpAffine(response_map, M, (h, w))
        if handedness == "R":
            response_map = np.fliplr(response_map)
        response_map = response_map.reshape(1, h, w) * 100  # scale to percent

        index.append([session[0], session[1], epoch_id])
        responses.append([response_map])
    
response_maps = pd.DataFrame(responses, index=pd.MultiIndex.from_tuples(index, names=['mouse_id', 'day', 'epoch_id']), columns=['response_map'])
response_maps

# plot response maps
nanmedian = lambda x: np.nanmedian(np.stack(x, axis=0), axis=0)
response_baseline = response_maps.query("day < 0").groupby(["mouse_id"]).agg(nanmedian)
response_maps["resp_baseline"] = response_maps["response_map"] - response_baseline["response_map"]
response_maps

In [ ]:
# get ROI dataset
areas_for_stats = [
    "MOs-medial", "MOs-lateral", "MOp", "SSp-ll", "SSp-ul", "SSp-nosemouth", "SSp-bfd", "SSp-tr",
    "VISp", "VIS-medial", "VISa", "VISrl", "RSP-anterior", "RSP-posterior"
]
window = np.arange(0,20) # 20 frames after grasp onset (1 second)
wf_param_id = 6
rows = []
index = []
for session, df_session in tqdm(df_distances.query("is_similar==True").groupby(["mouse_id", "day"])):
    # get keys for this day
    key = dict(mouse_id=session[0], days_from_stroke_norm=session[1])
    key = (wf.ImageProcessing2 * exp.DaysFromStrokeNorm & dict(**key, wf_param_id=wf_param_id)).fetch1("KEY")
    roi_keys = (
        wf.AllenSegmentation2.ROI & key & [f"roi_name = '{area}'" for area in areas_for_stats]
    ).fetch("KEY")
    key_ipsi = (pt.PawRecording.Hand & dict(**key, side='ipsi')).fetch1("KEY")

    
    # fetch widefield data
    frame_timestamps = (wf.Synchronisation & key).fetch1("frame_timestamps_blue")

    # get epoch starts
    handedness = (exp.Handedness & key).fetch1("handedness")
    restriction = df_session.reset_index().filter(["mouse_id", "day", "epoch_id"]).rename(columns={"day": "days_from_stroke_norm"})
    start_times, epoch_ids = (pt.MovementSegmentation.Epoch * exp.DaysFromStrokeNorm & restriction & key_ipsi).fetch("start_time", "epoch_id", order_by="epoch_id")

    # iterate over rois
    for roi_key in roi_keys:
        dff, n_pixels = (wf.AllenSegmentation2.ROI & roi_key).fetch1("dff", "n_pixels")
        if n_pixels < 0:
            continue
        # iterate over epochs
        for start_time, epoch_id in zip(start_times, epoch_ids):
            start_frame = np.searchsorted(frame_timestamps, start_time)
            dff_window = dff[start_frame + window].mean() * 100
            index.append([session[0], session[1], epoch_id, roi_key['roi_id']])
            rows.append([dff_window])

# create dataframe

roi_responses = pd.DataFrame(rows, index=pd.MultiIndex.from_tuples(index, names=['mouse_id', 'day', 'epoch_id', 'roi_name']), columns=['dff_window']).fillna(0)
roi_responses


In [ ]:
roi_responses_baseline = roi_responses.query("day < 0").groupby(["mouse_id", "roi_name"]).mean()
roi_responses["y"] = roi_responses["dff_window"] - roi_responses_baseline["dff_window"]
roi_responses_to_stat = roi_responses.join(stroke_groups, on='mouse_id').join(phases, on=['mouse_id', 'day'])
roi_responses_to_stat["g"] = pd.Categorical(roi_responses_to_stat["stroke_group"], categories=["Sham", "Stroke", "Stroke + training"], ordered=True)
roi_responses_to_stat["p"] = pd.Categorical(roi_responses_to_stat["phase"], categories=["Expert", "Early", "Late"], ordered=True)
roi_responses_to_stat["d"] = pd.Categorical(roi_responses_to_stat.index.get_level_values("day").astype(str), categories=[str(d) for d in [-3, -2, -1, 3, 7, 14, 21, 28]], ordered=True)

base = importr('base')
lme4 = importr('lme4')
emmeans = importr('emmeans')
stats = importr('stats')


results = []
for roi, df_roi in tqdm(roi_responses_to_stat.query("p!='Expert'").groupby('roi_name')):
    with (robjects.default_converter + pandas2ri.converter).context():
        model = lme4.lmer('y ~1 + g * p + (1|mouse_id/d)', data=df_roi.reset_index())
        formula = "pairwise ~ g | p"
        emm = emmeans.emmeans(model, stats.formula(formula), adjust="none")
        contrasts = base.summary(emm[1])
    
    contrasts["roi_name"] = roi
    results.append(contrasts)

df_results = pd.concat(results, ignore_index=True).set_index(["roi_name", "contrast", "p"])
df_results["p_adj"] = false_discovery_control(df_results["p.value"], method='bh')
df_results
    

In [ ]:
# output dataframe for manuscript submission
df_p_value_table = df_results.query("p_adj <= 0.05").reset_index().copy()
map_p_to_phase = {
    "Post Early":"Early",
    "Post Late":"Late"
}
# df_p_value_table["stroke_phase"] = df_p_value_table["p"].map(map_p_to_phase)
df_p_value_table["stroke_phase"] = df_p_value_table["p"].copy()
contrast_to_name = {
    "Sham - Stroke":"Sham vs. Stroke",
    "Sham - (Stroke + training)":"Sham vs. Stroke + training",
    "Stroke - (Stroke + training)":"Stroke vs. Stroke + training"
}
df_p_value_table["contrast_name"] = df_p_value_table["contrast"].map(contrast_to_name)
def format_p_value(p):
    if p < 0.001:
        return "< 0.001"
    else:
        return f"{p:.3f}"
def format_ROI(r):
    return r.replace('_contra', '_R').replace('_ipsi', '_L')

def get_comparison(row):
    return f"({row['roi_formatted']}) {row['stroke_phase']}: {row['contrast_name']}"

df_p_value_table["p_value_formatted"] = df_p_value_table["p_adj"].apply(format_p_value)
df_p_value_table["roi_formatted"] = df_p_value_table["roi_name"].apply(format_ROI)
df_p_value_table["comparison"] = df_p_value_table.apply(get_comparison, axis=1)
df_p_value_table = df_p_value_table[["comparison", "p_value_formatted"]]
df_p_value_table.to_csv(p_figures / f"p_value_table_{dataset_name}.csv")
display(df_p_value_table)

In [ ]:
nanmean = lambda x: np.nanmean(np.stack(x, axis=0), axis=0)
maps_to_compare = response_maps.join(stroke_groups, on='mouse_id').join(phases, on=['mouse_id', 'day']).groupby(["stroke_group", "phase"]).agg(nanmean)
maps_to_compare

areas = [
    "MOs", "MOp", "SSp-ll", "SSp-ul", "SSp-nosemouth", "SSp-bfd", "SSp-tr",
    "RSP", "VISp", "VIS-medial", "VISa", "VISrl",
]
areas_to_overlay = [a+"_R" for a in areas] + [a+"_L" for a in areas]
areas_to_dot = ["MOs-medial_R", "MOs-medial_L", "RSP-anterior_R", "RSP-anterior_L"]

masks, area_names, edges, mask_total, bregma = load_allen((128, 128))
mask_combined = np.zeros((128, 128), dtype=np.uint8)
for i in range(len(area_names)):
    if area_names[i] in areas_to_overlay + ["SSp-un_R", "SSp-un_L"]:
        mask_combined[masks[i]>0] = 255
# close holes
mask_combined = cv2.morphologyEx(mask_combined, cv2.MORPH_CLOSE, np.ones((5,5), np.uint8))
mask_combined = (mask_combined > 0).astype(np.uint8) * 255

phase_to_ax={'Early':0, 'Late':1}
vmin = -0.5
vmax = 0.5
cmap = 'PiYG'
f, ax = plt.subplots(3,2, figsize=(9*cm, 10*cm), sharex=True, sharey=True,
                     gridspec_kw=dict(wspace=0, hspace=0.15, left=0, right=0.95, top=0.9, bottom=0.05))

for phase, df_phase in maps_to_compare.groupby("phase"):
    if phase=='Expert':
        continue
    sham_vs_stroke = (df_phase.loc[("Stroke", phase), "resp_baseline"] - df_phase.loc[("Sham", phase), "resp_baseline"]).squeeze()
    sham_vs_rehab = (df_phase.loc[("Stroke + training", phase), "resp_baseline"] - df_phase.loc[("Sham", phase), "resp_baseline"]).squeeze()
    stroke_vs_rehab = (df_phase.loc[("Stroke + training", phase), "resp_baseline"] - df_phase.loc[("Stroke", phase), "resp_baseline"]).squeeze()
    sham_vs_stroke[mask_combined==0] = np.nan
    sham_vs_rehab[mask_combined==0] = np.nan
    stroke_vs_rehab[mask_combined==0] = np.nan
    # plot
    i = phase_to_ax[phase]
    imsh0 = ax[0, i].imshow(sham_vs_stroke, vmin=vmin, vmax=vmax, cmap=cmap)
    imsh1 = ax[1, i].imshow(sham_vs_rehab, vmin=vmin, vmax=vmax, cmap=cmap)
    imsh2 = ax[2, i].imshow(stroke_vs_rehab, vmin=vmin, vmax=vmax, cmap=cmap)
    # allen
    overlay_allen(ax[0, i], areas_to_overlay=areas_to_overlay, show_bregma=False, res=(128,128),
                  line_kw=dict(color='k', linewidth=0.5, alpha=0.5))
    overlay_allen(ax[1, i], areas_to_overlay=areas_to_overlay, show_bregma=False, res=(128,128),
                  line_kw=dict(color='k', linewidth=0.5, alpha=0.5))
    overlay_allen(ax[2, i], areas_to_overlay=areas_to_overlay, show_bregma=False, res=(128,128),
                  line_kw=dict(color='k', linewidth=0.5, alpha=0.5))
    # titles
    ax[0,i].set_xlim(10,118)
    ax[1,i].set_xlim(10,118)
    ax[2,i].set_xlim(10,118)

    significant_rois = df_results.query("p==@phase and p_adj<0.05").reset_index()
    for j, row in significant_rois.iterrows():
        # replace _contra
        # replace _contra with _R and _ipsi with _L
        area_to_star = row['roi_name'].replace('_contra', '_R').replace('_ipsi', '_L')
        # get mask for area
        mask_roi = masks[area_names == area_to_star].squeeze()
        # get coordinates of center of mass
        from scipy.ndimage import center_of_mass
        com = center_of_mass(mask_roi)
        p_value = row['p_adj']
        if p_value < 0.001:
            star = '***'
        elif p_value <= 0.01:
            star = '**'
        elif p_value <= 0.05:
            star = '*'
        # contrast
        if row['contrast'] == "Sham - Stroke":
            ax[0,i].text(com[1], com[0], star, color='b', fontsize=fontsize_small-1, ha='center', va='top')
        elif row['contrast'] == "Sham - (Stroke + training)":
            ax[1,i].text(com[1], com[0], star, color='b', fontsize=fontsize_small-1, ha='center', va='top')
        elif row['contrast'] == "Stroke - (Stroke + training)":
            ax[2,i].text(com[1], com[0], star, color='b', fontsize=fontsize_small-1, ha='center', va='top')

# pre, early, late
ax[-1,0].text(0.5, -0.1, "Early", fontsize=fontsize_medium, ha='center', va='bottom', transform=ax[-1,0].transAxes)
ax[-1,1].text(0.5, -0.1, "Late", fontsize=fontsize_medium, ha='center', va='bottom', transform=ax[-1,1].transAxes)
ax[0,0].text(0.5, 0.95, "Response differences between groups\nrelative to pre-stroke", fontsize=fontsize_medium, ha='center', va='top', transform=f.transFigure)

# colorbar text
cbar_0 = plt.colorbar(imsh0, ax=ax[0,:], label='$\\Delta$ F/F (%)', ticks=[vmin,0,vmax], orientation='vertical', shrink=0.7, aspect=10, pad=0.15)
cbar_0.ax.text(0.5, -0.07, "Sham > Stroke", fontsize=fontsize_small, ha='center', va='top', transform=cbar_0.ax.transAxes)
cbar_0.ax.text(0.5, 1.07, "Stroke > Sham", fontsize=fontsize_small, ha='center', va='bottom', transform=cbar_0.ax.transAxes)

cbar_1 = plt.colorbar(imsh0, ax=ax[1,:], label='$\\Delta$ F/F (%)', ticks=[vmin,0,vmax], orientation='vertical', shrink=0.7, aspect=10, pad=0.15)
cbar_1.ax.text(0.5, -0.07, "Sham > (Stroke + training)", fontsize=fontsize_small, ha='center', va='top', transform=cbar_1.ax.transAxes)
cbar_1.ax.text(0.5, 1.07, "(Stroke + training) > Sham", fontsize=fontsize_small, ha='center', va='bottom', transform=cbar_1.ax.transAxes)

cbar_2 = plt.colorbar(imsh0, ax=ax[2,:], label='$\\Delta$ F/F (%)', ticks=[vmin,0,vmax], orientation='vertical', shrink=0.7, aspect=10, pad=0.15)
cbar_2.ax.text(0.5, -0.07, "Stroke > (Stroke + training)", fontsize=fontsize_small, ha='center', va='top', transform=cbar_2.ax.transAxes)
cbar_2.ax.text(0.5, 1.07, "(Stroke + training) > Stroke", fontsize=fontsize_small, ha='center', va='bottom', transform=cbar_2.ax.transAxes)

f.savefig(p_figures / f"response_maps_differences_{dataset_name}.svg", dpi=600)
plt.show(f)
